## Module 3: Machine Learning for Classification

This module covers the fundamentals of **classification**, with a focus on predicting categorical outcomes from data. It introduces the typical classification workflow, including data preparation, feature analysis, model training, and evaluation. The main model used is **logistic regression**, along with techniques for handling categorical variables and interpreting model performance using metrics such as accuracy.

### Part 1: Data Preparation

In [1]:
# get telecom churn data
!wget 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv' -O data-week-3.csv

--2026-09-21 18:14:58--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  --.-KB/s    in 0.009s  

2026-09-21 18:14:58 (103 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [4]:
# load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# check the data
df = pd.read_csv('data-week-3.csv')
# since columns are too many and hidden, use transpose
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [5]:
# rewrite column names lowercase with no space (to be able to use in dot form)
df.columns = df.columns.str.lower().str.replace(' ', '_')

# check categorical columns and make their content lowercase with no space
cat_cols = list(df.dtypes[df.dtypes == 'str'].index)
for col in cat_cols:
    df[col] = df[col].str.lower().str.replace(' ', '_')

df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [ ]:
# check data types to clean the data 
# (totalcharges should be numeric but str, churn should be 1 or 0)
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [25]:
# turn column into numeric and ignore errors ('_' ones), fill missing with 0 (although not the best option)
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce').fillna(0)

# turn column into numeric rather than yes or no
df.churn = (df.churn == 'yes').astype(int)

### Part 2: Setting Up the Validation Framework

In [ ]:
from sklearn.model_selection import train_test_split

# split data with sckit-learn (since sckitlearn func divides 2, do 2 times)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1) # 1/5
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1) # 1/4 of remaining 4/5

# reset index for each
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# separate target variable
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

# delete target variable data from features
del df_train['churn']
del df_val['churn']
del df_test['churn']

# get the size of each set
print("Total size:", len(df))
print(f"Train, Val, Test: ({len(df_train)}, {len(df_val)}, {len(df_test)})")
print(f"y_train, y_val, y_test: ({len(y_train)}, {len(y_val)}, {len(y_test)})")

Total size: 7043
Train, Val, Test: (4225, 1409, 1409)
y_train, y_val, y_test: (4225, 1409, 1409)


### Part 3: Exploratory Data Analysis (EDA)

In [27]:
# use data except test one, check for null values
df_full_train = df_full_train.reset_index(drop=True)
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [28]:
# check mean and distribution of target variable
print('Mean churn (churn rate of customers):', df_full_train.churn.mean())
df_full_train.churn.value_counts(normalize=True)

Mean churn (churn rate of customers): 0.26996805111821087


churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [ ]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]
df_full_train[categorical].nunique()

### Part 4: Feature Importance - Churn Rate and Risk Ratio
#### Churn Rate:

In [ ]:
df_full_train.head()

In [ ]:
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_female

In [ ]:
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()
churn_male

In [ ]:
global_churn = df_full_train.churn.mean()
global_churn

In [ ]:
global_churn - churn_female

In [ ]:
global_churn - churn_male

In [ ]:
df_full_train.partner.value_counts()

In [ ]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_partner

In [ ]:
global_churn - churn_partner

In [ ]:
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()
churn_no_partner

In [ ]:
global_churn - churn_no_partner

#### Risk Ratio:

In [ ]:
churn_no_partner / global_churn

In [ ]:
churn_partner / global_churn

In [ ]:
from IPython.display import display

In [ ]:
for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)
    print()
    print()

### Part 5: Feature Importance - Mutual Information

### Part 6: One-hot Encoding

### Part 7: Logistic Regression

### Part 8: Training Logistic Regression with Scikit-Learn

### Part 9: Model Interpretation

### Part 10: Using the Model